# Student Progress Tracker with Question Graph Visualization

This notebook provides comprehensive student progress tracking with interactive graph visualization. Track:
- **Overall Progress**: Tests attempted, pass rate, average scores, trends
- **Question Performance**: Success rates, problem areas, common mistakes
- **Interactive Graph**: Questions as nodes (green=correct, red=wrong) with opacity animation
- **Test History Review**: Revisit past tests, review answers, track improvement
- **Simulation Engine**: Watch tests replay as nodes light up during test progression

## Features:
- 📊 Progress dashboard with metrics and trends
- 🎯 Per-question success analysis
- 🔗 Interactive networkx graph with plotly rendering
- ⏱️ Opacity transitions (50% → 100%) as questions are visited
- 🎬 Simulation engine to replay test attempts
- 📝 Review system for past attempts with detailed feedback


In [79]:


# ============================================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================================

import json
import uuid
from pathlib import Path
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import matplotlib.colors as mcolors
import networkx as nx
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import requests
from IPython.display import display, HTML
import ipywidgets as widgets
from ipywidgets import HBox, VBox, Label, Output, FloatSlider, IntSlider, Dropdown, Button, interact

# Configure display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ All libraries imported successfully")



✅ All libraries imported successfully


## Configuration

Set your Supabase credentials and student ID to load data. You can provide a JWT token for authentication.


In [80]:


# ============================================================
# 2. CONFIGURATION & AUTHENTICATION
# ============================================================

ROOT = Path.cwd()
ENV_PATH = ROOT / ".env"

def read_env_file(path: Path) -> dict:
    env = {}
    if not path.exists():
        print(f"⚠️ .env file not found at {path}")
        return env
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        env[key.strip()] = value.strip().strip('"').strip("'")
    return env

ENV = read_env_file(ENV_PATH)
SUPABASE_URL = ENV.get("VITE_STUDENT_SUPABASE_URL")
SUPABASE_KEY = ENV.get("VITE_STUDENT_SUPABASE_PUBLISHABLE_KEY")

# Set your credentials below
SUPABASE_ACCESS_TOKEN = ""  # Paste a valid JWT token here, or leave blank to use public key
STUDENT_ID = ""  # Enter student UUID here

if not SUPABASE_URL or not SUPABASE_KEY:
    print("⚠️ Missing Supabase credentials in .env file")
else:
    print(f"✅ Supabase URL: {SUPABASE_URL[:50]}...")
    print(f"✅ Supabase Key loaded")

# Function to fetch data from Supabase with retry logic
def fetch_from_supabase(table: str, query_filter: str = None, token: str = None, retries: int = 3, verbose: bool = False) -> List[Dict]:
    """Fetch data from Supabase with error handling and retries"""
    import time
    
    if not SUPABASE_URL or not SUPABASE_KEY:
        print("❌ Supabase credentials not configured")
        return []
    
    for attempt in range(retries):
        try:
            url = f"{SUPABASE_URL}/rest/v1/{table}"
            headers = {
                "apikey": SUPABASE_KEY,
                "Content-Type": "application/json",
                "Prefer": "return=representation"
            }
            if token:
                headers["Authorization"] = f"Bearer {token}"
            
            if query_filter:
                url += f"?{query_filter}"
            else:
                url += "?limit=1000"
            
            if verbose:
                print(f"\n📡 Fetching {table} (attempt {attempt + 1}/{retries}):")
                print(f"   URL: {url}")
                print(f"   Headers: apikey={SUPABASE_KEY[:15]}..., auth={'Bearer ...' if token else 'none'}")
            
            response = requests.get(url, headers=headers, timeout=15)
            
            if verbose:
                print(f"   Response: {response.status_code}")
            
            if response.status_code == 200:
                data = response.json()
                if verbose:
                    print(f"   ✅ Success: Got {len(data)} record(s)")
                return data
            elif response.status_code == 401:
                print(f"❌ RLS Policy: Unauthorized access to {table}")
                print(f"   This usually means:")
                print(f"   • You need a JWT token in SUPABASE_ACCESS_TOKEN")
                print(f"   • Or RLS policies need to allow public key access")
                return []
            else:
                error_text = response.text[:200] if response.text else "No details"
                print(f"❌ Error {response.status_code} fetching {table}")
                print(f"   Response: {error_text}")
                if attempt < retries - 1:
                    wait_time = 1 + (attempt * 0.5)
                    print(f"   Retrying in {wait_time}s ({attempt + 1}/{retries})...")
                    time.sleep(wait_time)
                else:
                    return []
        except requests.exceptions.ConnectionError as e:
            print(f"❌ CONNECTION ERROR fetching {table}:")
            print(f"   {str(e)}")
            print(f"   Check that SUPABASE_URL is correct: {SUPABASE_URL[:50]}...")
            if attempt < retries - 1:
                wait_time = 2 + (attempt * 1)
                print(f"   Retrying in {wait_time}s...")
                time.sleep(wait_time)
        except requests.exceptions.Timeout:
            print(f"❌ Timeout fetching {table} (attempt {attempt + 1}/{retries})")
            if attempt < retries - 1:
                time.sleep(2)
        except Exception as e:
            print(f"❌ Error fetching {table}: {type(e).__name__}: {str(e)}")
            if attempt < retries - 1:
                time.sleep(1)
    
    return []

print("✅ Configuration loaded")



✅ Supabase URL: https://ukiuxecvybwvngwirjqt.supabase.co...
✅ Supabase Key loaded
✅ Configuration loaded


## Debugging Connection Issues

Before loading test data, verify your Supabase connection and credentials are correct.



In [81]:


# ============================================================
# 2.5 VERIFY SUPABASE CONNECTION
# ============================================================

print("🔍 DIAGNOSING SUPABASE CONNECTION\n")
STUDENT_ID="12efa469-0330-42e1-bc64-82bed3402ae8"

# Step 1: Check credentials are loaded
if not SUPABASE_URL or not SUPABASE_KEY:
    print("❌ ERROR: Supabase credentials not loaded from .env")
    print("   Make sure .env file exists with:")
    print("   VITE_STUDENT_SUPABASE_URL=https://ukiuxecvybwvngwirjqt.supabase.co")
    print("   VITE_STUDENT_SUPABASE_PUBLISHABLE_KEY=sb_publishable_uaXOE8LGq_WvpiSEwfAOGQ_yLbC9uob")
else:
    print(f"✅ SUPABASE_URL loaded: {SUPABASE_URL[:40]}...")
    print(f"✅ SUPABASE_KEY loaded: {SUPABASE_KEY[:20]}...")

# Step 2: Check if student ID is set
if not STUDENT_ID:
    print("\n⚠️  WARNING: STUDENT_ID is empty")
    print("   Set it in cell 3 or update the code above")
else:
    print(f"\n✅ STUDENT_ID set: {STUDENT_ID}")

# Step 3: Test connection with simple query
print("\n📡 Testing API connection...")
try:
    url = f"{SUPABASE_URL}/rest/v1/test_history?limit=1"
    headers = {
        "apikey": SUPABASE_KEY,
        "Content-Type": "application/json",
    }
    
    print(f"\n   Request URL: {url}")
    print(f"   Headers: {{'apikey': '{SUPABASE_KEY[:20]}...', 'Content-Type': 'application/json'}}")
    
    response = requests.get(url, headers=headers, timeout=10)
    
    print(f"\n   Response Status: {response.status_code}")
    
    if response.status_code == 200:
        data = response.json()
        print(f"   ✅ Connection successful! Got {len(data)} records")
        if len(data) > 0:
            print(f"   Sample record keys: {list(data[0].keys())}")
    elif response.status_code == 401:
        print(f"   ❌ UNAUTHORIZED (401)")
        print(f"   Reason: Invalid API key or RLS policy blocking access")
        print(f"   Response: {response.text[:200]}")
    else:
        print(f"   ❌ Error {response.status_code}")
        print(f"   Response: {response.text[:200]}")
        
except requests.exceptions.ConnectionError as e:
    print(f"   ❌ CONNECTION ERROR: {str(e)}")
    print(f"   \n   This usually means:")
    print(f"   • Supabase URL is invalid")
    print(f"   • Network is blocked")
    print(f"   • Firewall/proxy issue")
    print(f"   \n   Check that SUPABASE_URL is correct (should be https://xxx.supabase.co)")
    
except requests.exceptions.Timeout as e:
    print(f"   ❌ TIMEOUT: {str(e)}")
    print(f"   Request took too long. Check your internet connection.")
    
except Exception as e:
    print(f"   ❌ UNEXPECTED ERROR: {type(e).__name__}: {str(e)}")

print("\n" + "="*60)



🔍 DIAGNOSING SUPABASE CONNECTION

✅ SUPABASE_URL loaded: https://ukiuxecvybwvngwirjqt.supabase.co...
✅ SUPABASE_KEY loaded: sb_publishable_uaXOE...

✅ STUDENT_ID set: 12efa469-0330-42e1-bc64-82bed3402ae8

📡 Testing API connection...

   Request URL: https://ukiuxecvybwvngwirjqt.supabase.co/rest/v1/test_history?limit=1
   Headers: {'apikey': 'sb_publishable_uaXOE...', 'Content-Type': 'application/json'}

   Response Status: 200
   ✅ Connection successful! Got 0 records



In [82]:


# ============================================================
# 3. LOAD AND PARSE TEST DATA
# ============================================================
STUDENT_ID="12efa469-0330-42e1-bc64-82bed3402ae8"

def load_student_test_history(student_id: str, token: str = None, verbose: bool = True) -> pd.DataFrame:
    """Load test history for a specific student"""
    if not student_id:
        print("❌ Student ID not set")
        return pd.DataFrame()
    
    try:
        filter_str = f"user_id=eq.{student_id}&order=completed_at.desc&limit=1000"
        data = fetch_from_supabase("test_history", filter_str, token, verbose=verbose)
        
        if not data:
            print(f"\n❌ No test history found for student {student_id}")
            print(f"\nNext steps to debug:")
            print(f"1. Verify the student UUID is correct")
            print(f"2. Check if tests exist in database (any student):")
            print(f"   - Go to Supabase console → SQL Editor")
            print(f"   - Run: SELECT COUNT(*) FROM test_history;")
            print(f"3. If tests exist for other students, check RLS policies:")
            print(f"   - Go to test_history table → RLS")
            print(f"   - Add policy: INSERT/SELECT for authenticated users")
            print(f"4. If no tests exist, run a test in the web app first")
            return pd.DataFrame()
        
        df = pd.DataFrame(data)
        df['completed_at'] = pd.to_datetime(df['completed_at'])
        
        print(f"\n✅ Loaded {len(df)} test attempts from database")
        print(f"   Columns: {list(df.columns)}")
        if 'review_payload' in df.columns:
            has_payload = df['review_payload'].notna().sum()
            print(f"   Tests with review_payload: {has_payload}/{len(df)}")
            parsed_payloads = df['review_payload'].apply(parse_review_payload)
            counted_tests = sum(1 for payload in parsed_payloads if payload.get('counts_for_stats', True))
            reviewable_tests = sum(1 for payload in parsed_payloads if payload.get('question_ids'))
            print(f"   Counted in stats: {counted_tests}/{len(df)}")
            print(f"   Review-ready attempts: {reviewable_tests}/{len(df)}")
        return df
    except Exception as e:
        print(f"\n❌ Error loading test history: {type(e).__name__}")
        print(f"   {str(e)}")
        return pd.DataFrame()

def parse_review_payload(review_payload: dict) -> dict:
    """Parse review payload from test history"""
    base_payload = {
        'question_ids': [],
        'answers': [],
        'question_reviews': [],
        'question_snapshots': [],
        'attempt_kind': None,
        'counts_for_stats': True,
        'counts_for_rating': True,
        'warning_breakdown': {},
        'review_metadata': {},
        'source_attempt_id': None,
    }
    
    if not review_payload or not isinstance(review_payload, dict):
        return base_payload
    
    question_ids = [q_id for q_id in review_payload.get('question_ids', []) if isinstance(q_id, str)]
    answers = review_payload.get('answers', []) if isinstance(review_payload.get('answers', []), list) else []
    question_reviews = [
        item for item in review_payload.get('question_reviews', [])
        if isinstance(item, dict)
    ] if isinstance(review_payload.get('question_reviews', []), list) else []
    question_snapshots = [
        item for item in review_payload.get('question_snapshots', [])
        if isinstance(item, dict)
    ] if isinstance(review_payload.get('question_snapshots', []), list) else []
    
    if not question_ids and question_snapshots:
        question_ids = [item.get('id') for item in question_snapshots if isinstance(item.get('id'), str)]
    
    warning_breakdown = review_payload.get('warningBreakdown', {})
    if not isinstance(warning_breakdown, dict):
        warning_breakdown = {}
    
    review_metadata = review_payload.get('reviewMetadata', {})
    if not isinstance(review_metadata, dict):
        review_metadata = {}
    
    source_attempt_id = review_metadata.get('source_attempt_id')
    if source_attempt_id is not None and not isinstance(source_attempt_id, str):
        source_attempt_id = None
    
    return {
        'question_ids': question_ids,
        'answers': answers,
        'question_reviews': question_reviews,
        'question_snapshots': question_snapshots,
        'attempt_kind': review_payload.get('attemptKind') if isinstance(review_payload.get('attemptKind'), str) else None,
        'counts_for_stats': bool(review_payload.get('countsForStats', True)),
        'counts_for_rating': bool(review_payload.get('countsForRating', True)),
        'warning_breakdown': warning_breakdown,
        'review_metadata': review_metadata,
        'source_attempt_id': source_attempt_id,
    }

def _is_answered_value(answer_value):
    if answer_value is None:
        return False
    if isinstance(answer_value, list):
        return len(answer_value) > 0
    if isinstance(answer_value, str):
        return answer_value.strip() != ""
    return True

def infer_correctness(question_snapshot: dict, answer_value, review_data: dict) -> bool:
    if review_data and isinstance(review_data.get('correct'), bool):
        return review_data.get('correct', False)
    
    if not question_snapshot or not isinstance(question_snapshot, dict):
        return False
    
    if not _is_answered_value(answer_value):
        return False
    
    question_type = question_snapshot.get('type')
    if question_type == 'mcq':
        return answer_value == question_snapshot.get('correctAnswer')
    
    if question_type == 'msq':
        correct_answers = question_snapshot.get('correctAnswers', [])
        if not isinstance(answer_value, list) or not isinstance(correct_answers, list):
            return False
        return sorted(answer_value) == sorted(correct_answers)
    
    if question_type == 'nat':
        correct_nat = question_snapshot.get('correctNat')
        if not isinstance(correct_nat, dict):
            return False
        try:
            numeric_value = float(answer_value)
        except (TypeError, ValueError):
            return False
        return correct_nat.get('min', numeric_value + 1) <= numeric_value <= correct_nat.get('max', numeric_value - 1)
    
    return False

def extract_question_performance(test_history_df: pd.DataFrame) -> pd.DataFrame:
    """Extract per-question performance from test history"""
    all_question_data = []
    
    for idx, row in test_history_df.iterrows():
        review_payload = parse_review_payload(row.get('review_payload', {}))
        question_ids = review_payload.get('question_ids', [])
        answers = review_payload.get('answers', [])
        question_reviews = review_payload.get('question_reviews', [])
        question_snapshots = review_payload.get('question_snapshots', [])
        warning_breakdown = review_payload.get('warning_breakdown', {})
        review_lookup = {
            review.get('questionId'): review
            for review in question_reviews
            if isinstance(review.get('questionId'), str)
        }
        focus_warning_count = warning_breakdown.get('violations', row.get('violations', 0))
        rapid_guess_total = sum(1 for review in question_reviews if review.get('rapidGuessWarning'))
        
        for q_idx, q_id in enumerate(question_ids):
            review_data = review_lookup.get(q_id, {})
            question_snapshot = question_snapshots[q_idx] if q_idx < len(question_snapshots) else {}
            answer_value = answers[q_idx] if q_idx < len(answers) else None
            correct = infer_correctness(question_snapshot, answer_value, review_data)
            
            all_question_data.append({
                'test_id': row['id'],
                'question_id': q_id,
                'correct': correct,
                'time_spent_seconds': review_data.get('timeSpentSeconds', 0) if review_data else 0,
                'rapid_guess': review_data.get('rapidGuessWarning', False) if review_data else False,
                'warning_text': review_data.get('warningText') if review_data else None,
                'elo_adjustment': review_data.get('eloAdjustment', 0) if review_data else 0,
                'selected_answer': answer_value,
                'test_type': row['test_type'],
                'attempt_kind': review_payload.get('attempt_kind') or row['test_type'],
                'counts_for_stats': review_payload.get('counts_for_stats', True),
                'counts_for_rating': review_payload.get('counts_for_rating', True),
                'focus_warning_count': focus_warning_count,
                'rapid_guess_total': rapid_guess_total,
                'question_subject_id': question_snapshot.get('subjectId'),
                'question_topic_id': question_snapshot.get('topicId'),
                'question_type': question_snapshot.get('type'),
                'question_difficulty': question_snapshot.get('difficulty'),
                'question_marks': question_snapshot.get('marks'),
                'source_attempt_id': review_payload.get('source_attempt_id'),
                'completed_at': row['completed_at'],
                'score': row['score'],
                'max_score': row['max_score'],
            })
    
    if not all_question_data:
        return pd.DataFrame()
    
    return pd.DataFrame(all_question_data)

# Load data
print("="*60)
print("📊 LOADING STUDENT TEST DATA")
print("="*60)

if STUDENT_ID:
    test_history = load_student_test_history(STUDENT_ID, SUPABASE_ACCESS_TOKEN, verbose=True)
    question_performance = extract_question_performance(test_history)
    
    if not test_history.empty:
        print(f"\n📊 Test History Summary:")
        print(f"  - Total tests: {len(test_history)}")
        print(f"  - Unique questions attempted: {question_performance['question_id'].nunique()}")
        print(f"  - Average score: {test_history['score'].mean():.1f}/{test_history['max_score'].iloc[0] if len(test_history) > 0 else 0}")
    else:
        test_history = pd.DataFrame()
        question_performance = pd.DataFrame()
else:
    print("❌ STUDENT_ID not set - cannot load data")
    print("   Update STUDENT_ID in cell 3 above (should be a UUID string)")
    test_history = pd.DataFrame()
    question_performance = pd.DataFrame()



📊 LOADING STUDENT TEST DATA

📡 Fetching test_history (attempt 1/3):
   URL: https://ukiuxecvybwvngwirjqt.supabase.co/rest/v1/test_history?user_id=eq.12efa469-0330-42e1-bc64-82bed3402ae8&order=completed_at.desc&limit=1000
   Headers: apikey=sb_publishable_..., auth=none
   Response: 200
   ✅ Success: Got 0 record(s)

❌ No test history found for student 12efa469-0330-42e1-bc64-82bed3402ae8

Next steps to debug:
1. Verify the student UUID is correct
2. Check if tests exist in database (any student):
   - Go to Supabase console → SQL Editor
   - Run: SELECT COUNT(*) FROM test_history;
3. If tests exist for other students, check RLS policies:
   - Go to test_history table → RLS
   - Add policy: INSERT/SELECT for authenticated users
4. If no tests exist, run a test in the web app first


## Student Progress Tracking Dashboard

Interactive dashboard showing overall progress metrics, test trends, and improvement indicators.


In [83]:


# ============================================================
# 4. STUDENT PROGRESS TRACKING DASHBOARD
# ============================================================

def create_progress_dashboard(test_history_df: pd.DataFrame, question_perf_df: pd.DataFrame):
    """Create interactive progress dashboard"""
    
    if test_history_df.empty:
        print("⚠️ No test data available to create dashboard")
        return
    
    # Calculate metrics
    total_tests = len(test_history_df)
    avg_score = test_history_df['score'].mean()
    max_score_first = test_history_df['max_score'].iloc[0] if len(test_history_df) > 0 else 100
    accuracy = (question_perf_df['correct'].sum() / len(question_perf_df) * 100) if len(question_perf_df) > 0 else 0
    unique_questions = question_perf_df['question_id'].nunique() if len(question_perf_df) > 0 else 0
    correct_count = question_perf_df['correct'].sum() if len(question_perf_df) > 0 else 0
    
    # Score trend
    test_history_sorted = test_history_df.sort_values('completed_at')
    
    # Create subplots (3 panels, not 4 - pie chart needs separate handling)
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=("Score Trend Over Time", "Accuracy by Test Type", 
                       "Questions Summary", "Test Count"),
        specs=[[{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": False}, {"secondary_y": False}]]
    )
    
    # Plot 1: Score Trend
    fig.add_trace(
        go.Scatter(x=test_history_sorted['completed_at'], 
                  y=test_history_sorted['score'],
                  mode='lines+markers',
                  name='Score',
                  line=dict(color='#3b82f6', width=2)),
        row=1, col=1
    )
    
    # Plot 2: Accuracy by Test Type
    if not question_perf_df.empty:
        accuracy_by_type = question_perf_df.groupby('test_type')['correct'].agg(['sum', 'count'])
        accuracy_by_type['percentage'] = (accuracy_by_type['sum'] / accuracy_by_type['count'] * 100)
        
        fig.add_trace(
            go.Bar(x=accuracy_by_type.index, y=accuracy_by_type['percentage'],
                  name='Accuracy %',
                  marker=dict(color='#10b981')),
            row=1, col=2
        )
    
    # Plot 3: Correct vs Wrong (stacked bar instead of pie)
    if len(question_perf_df) > 0:
        correct_count = question_perf_df['correct'].sum()
        wrong_count = len(question_perf_df) - correct_count
        
        fig.add_trace(
            go.Bar(x=['Questions'],
                   y=[correct_count],
                   name='Correct',
                   marker=dict(color='#10b981')),
            row=2, col=1
        )
        fig.add_trace(
            go.Bar(x=['Questions'],
                   y=[wrong_count],
                   name='Wrong',
                   marker=dict(color='#ef4444')),
            row=2, col=1
        )
    
    # Plot 4: Test statistics
    fig.add_trace(
        go.Bar(x=['Total Tests', 'Avg Questions', 'Pass Rate'],
               y=[total_tests, len(question_perf_df) / max(total_tests, 1), accuracy],
               marker=dict(color=['#8b5cf6', '#3b82f6', '#10b981']),
               name='Stats'),
        row=2, col=2
    )
    
    # Update layout
    fig.update_layout(height=800, showlegend=True, title_text="Student Progress Dashboard", barmode='stack')
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_yaxes(title_text="Score", row=1, col=1)
    fig.update_xaxes(title_text="Test Type", row=1, col=2)
    fig.update_yaxes(title_text="Accuracy %", row=1, col=2)
    fig.update_xaxes(title_text="Category", row=2, col=1)
    fig.update_yaxes(title_text="Count", row=2, col=1)
    fig.update_xaxes(title_text="Metric", row=2, col=2)
    fig.update_yaxes(title_text="Value", row=2, col=2)
    
    # Display metrics
    print("\n" + "="*60)
    print("📊 STUDENT PROGRESS DASHBOARD")
    print("="*60)
    print(f"Total Tests Attempted:     {total_tests}")
    print(f"Average Score:             {avg_score:.1f}/{max_score_first}")
    print(f"Overall Accuracy:          {accuracy:.1f}%")
    print(f"Unique Questions Attempted: {unique_questions}")
    print(f"Correct Answers:           {correct_count}")
    print(f"Wrong Answers:             {len(question_perf_df) - correct_count if len(question_perf_df) > 0 else 0}")
    print("="*60 + "\n")
    
    fig.show()

# Create dashboard
if not test_history.empty:
    create_progress_dashboard(test_history, question_performance)



## Question Performance Analysis

Detailed analysis of individual question performance, including success rates, attempt frequency, and problem areas.


In [84]:

# ============================================================
# 5. QUESTION PERFORMANCE ANALYSIS
# ============================================================

def analyze_question_performance(question_perf_df: pd.DataFrame) -> pd.DataFrame:
    """Analyze performance for each question"""
    
    if question_perf_df.empty:
        return pd.DataFrame()
    
    analysis = question_perf_df.groupby('question_id').agg({
        'correct': ['sum', 'count'],
        'time_spent_seconds': 'mean',
        'rapid_guess': 'sum'
    }).reset_index()
    
    analysis.columns = ['question_id', 'correct_count', 'total_attempts', 'avg_time_seconds', 'rapid_guess_count']
    analysis['success_rate'] = (analysis['correct_count'] / analysis['total_attempts'] * 100).round(1)
    analysis = analysis.sort_values('success_rate')
    
    return analysis

def display_question_analysis(analysis_df: pd.DataFrame):
    """Display question analysis"""
    
    if analysis_df.empty:
        print("⚠️ No question performance data available")
        return
    
    print("\n" + "="*80)
    print("📊 QUESTION PERFORMANCE ANALYSIS")
    print("="*80)
    
    # Top struggling questions
    print("\n🔴 Top Struggling Questions (Lowest Success Rate):")
    print("-" * 80)
    struggling = analysis_df.head(10)
    for idx, row in struggling.iterrows():
        print(f"  {row['question_id']:20} | Success: {row['success_rate']:6.1f}% | Attempts: {int(row['total_attempts']):3} | Avg Time: {row['avg_time_seconds']:6.1f}s")
    
    # Top performing questions
    print("\n🟢 Top Performing Questions (Highest Success Rate):")
    print("-" * 80)
    performing = analysis_df.tail(10)
    for idx, row in performing.iterrows():
        print(f"  {row['question_id']:20} | Success: {row['success_rate']:6.1f}% | Attempts: {int(row['total_attempts']):3} | Avg Time: {row['avg_time_seconds']:6.1f}s")
    
    print("="*80 + "\n")
    
    # Create visualization
    fig = go.Figure()
    
    # Sort by success rate
    sorted_df = analysis_df.sort_values('success_rate')
    
    fig.add_trace(go.Bar(
        x=sorted_df['success_rate'],
        y=sorted_df['question_id'],
        orientation='h',
        marker=dict(
            color=sorted_df['success_rate'],
            colorscale='RdYlGn',
            showscale=True,
            colorbar=dict(title="Success %")
        ),
        text=sorted_df['success_rate'].apply(lambda x: f"{x:.0f}%"),
        textposition='auto'
    ))
    
    fig.update_layout(
        title="Question Success Rate Analysis",
        xaxis_title="Success Rate (%)",
        yaxis_title="Question ID",
        height=max(400, len(sorted_df) * 20),
        margin=dict(l=150)
    )
    
    fig.show()

# Analyze and display
if not question_performance.empty:
    question_analysis = analyze_question_performance(question_performance)
    display_question_analysis(question_analysis)


## Interactive Graph Visualization with Opacity Animation

Build an interactive networkx graph visualization with:
- **Questions as nodes** with proper labeling
- **Color coding**: Green for correct answers, Red for incorrect/unattempted
- **Opacity animation**: 50% opacity initially, 100% when visited
- **Simulation engine**: Watch nodes light up as test progresses


In [85]:

# ============================================================
# 6. INTERACTIVE GRAPH VISUALIZATION WITH OPACITY ANIMATION
# ============================================================

def create_question_graph(question_perf_df: pd.DataFrame) -> nx.Graph:
    """Create a networkx graph with questions as nodes"""
    
    G = nx.Graph()
    
    if question_perf_df.empty:
        return G
    
    # Add nodes for each question
    for _, row in question_perf_df.iterrows():
        question_id = row['question_id']
        is_correct = row['correct']
        
        G.add_node(
            question_id,
            correct=is_correct,
            test_type=row['test_type'],
            time_spent=row['time_spent_seconds'],
            rapid_guess=row['rapid_guess']
        )
    
    # Create edges between questions in the same test
    test_groups = question_perf_df.groupby('test_id')
    for test_id, group in test_groups:
        questions = group['question_id'].tolist()
        for i in range(len(questions) - 1):
            G.add_edge(questions[i], questions[i + 1])
    
    return G

def create_plotly_graph_visualization(G: nx.Graph, title: str = "Question Performance Graph"):
    """Create an interactive Plotly visualization of the question graph"""
    
    if len(G.nodes()) == 0:
        print("⚠️ Graph has no nodes to visualize")
        return
    
    # Use spring layout for better visualization
    pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
    
    # Prepare edge traces
    edge_x = []
    edge_y = []
    for edge in G.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])
    
    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        mode='lines',
        line=dict(width=0.5, color='#999'),
        hoverinfo='none',
        showlegend=False
    )
    
    # Prepare node traces with color and opacity based on correctness
    node_x = []
    node_y = []
    node_text = []
    node_color = []
    node_opacity = []
    node_size = []
    node_hover = []
    
    for node in G.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        
        # Get node attributes
        is_correct = G.nodes[node].get('correct', False)
        time_spent = G.nodes[node].get('time_spent', 0)
        rapid_guess = G.nodes[node].get('rapid_guess', False)
        
        # Determine color: Green for correct, Red for wrong, Gray for unattempted
        if is_correct:
            color = '#10b981'  # Green
        else:
            color = '#ef4444'  # Red
        
        node_color.append(color)
        node_opacity.append(1.0)  # Will be 1.0 for visited, but we'll set initial to 0.5 in simulation
        node_size.append(30)
        
        # Extract question number from ID (e.g., 'da-2025-q001' -> '1')
        q_num = node.split('-')[-1] if '-' in node else node
        node_text.append(q_num)
        
        # Hover text
        status = "✅ Correct" if is_correct else "❌ Wrong"
        hover_text = f"<b>{node}</b><br>Status: {status}<br>Time: {time_spent:.0f}s"
        if rapid_guess:
            hover_text += "<br>⚡ Rapid guess"
        node_hover.append(hover_text)
    
    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers+text',
        text=node_text,
        textposition='middle center',
        textfont=dict(size=12, color='white', family='Arial Black'),
        hovertext=node_hover,
        hoverinfo='text',
        marker=dict(
            size=node_size,
            color=node_color,
            opacity=0.5,  # Initial opacity 50%
            line=dict(color='#fff', width=2)
        ),
        showlegend=False
    )
    
    # Create figure
    fig = go.Figure(data=[edge_trace, node_trace])
    
    fig.update_layout(
        title=title,
        showlegend=False,
        hovermode='closest',
        margin=dict(b=20, l=5, r=5, t=40),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        height=800,
        paper_bgcolor='#f8f9fa',
        plot_bgcolor='#f8f9fa'
    )
    
    fig.show()

def create_student_progress_network(question_perf_df: pd.DataFrame) -> nx.Graph:
    """Build a richer student graph with subject hubs, topic hubs, and question nodes"""
    
    G = nx.Graph()
    if question_perf_df.empty:
        return G
    
    latest_by_question = (
        question_perf_df
        .sort_values('completed_at')
        .drop_duplicates('question_id', keep='last')
    )
    
    for _, row in latest_by_question.iterrows():
        subject_id = row.get('question_subject_id') or row.get('test_type') or 'general'
        topic_id = row.get('question_topic_id') or row.get('question_id')
        subject_node = f"subject::{subject_id}"
        topic_node = f"topic::{topic_id}"
        question_node = f"question::{row['question_id']}"
        
        if not G.has_node(subject_node):
            G.add_node(
                subject_node,
                node_type='subject',
                label=str(subject_id).replace('-', ' ').title(),
                color='#facc15',
                size=34,
                title=f"Subject hub: {subject_id}",
            )
        
        if not G.has_node(topic_node):
            G.add_node(
                topic_node,
                node_type='topic',
                label=str(topic_id).replace('-', ' ').title(),
                color='#d1d5db',
                size=max(18, min(32, 14 + int(row.get('attempts', 1) or 1))),
                title=f"Topic hub: {topic_id}",
            )
        
        is_correct = bool(row.get('correct', False))
        color = '#22c55e' if is_correct else '#ef4444'
        label = str(row['question_id']).split('-')[-1].upper()[:12]
        G.add_node(
            question_node,
            node_type='question',
            label=label,
            color=color,
            size=12,
            title=(
                f"Question: {row['question_id']}\n"
                f"Status: {'Correct' if is_correct else 'Wrong'}\n"
                f"Subject: {subject_id}\n"
                f"Topic: {topic_id}\n"
                f"Time spent: {row.get('time_spent_seconds', 0)}s"
            ),
        )
        
        G.add_edge(subject_node, topic_node, edge_type='subject-topic', color='rgba(250,204,21,0.22)', width=1.1)
        G.add_edge(topic_node, question_node, edge_type='topic-question', color='rgba(148,163,184,0.20)', width=0.8)
    
    for test_id, group in question_perf_df.sort_values('completed_at').groupby('test_id'):
        question_nodes = [f"question::{question_id}" for question_id in group['question_id'].tolist()]
        for left, right in zip(question_nodes, question_nodes[1:]):
            if left in G and right in G:
                G.add_edge(left, right, edge_type='test-path', color='rgba(34,197,94,0.35)', width=1.6)
    
    return G

def render_interactive_student_graph(G: nx.Graph, title: str = 'Student Progress Graph', output_path: str = 'student_progress_graph.html'):
    """Render an interactive dark graph inspired by graph.webp"""
    
    if len(G.nodes()) == 0:
        print('No graph data available for visualization')
        return None
    
    nodes_payload = []
    edges_payload = []
    
    for node_id, data in G.nodes(data=True):
        node_type = data.get('node_type', 'question')
        font_size = 18 if node_type == 'subject' else 13 if node_type == 'topic' else 10
        nodes_payload.append({
            'id': node_id,
            'label': data.get('label', node_id),
            'title': data.get('title', data.get('label', node_id)),
            'shape': 'dot',
            'size': data.get('size', 12),
            'color': {
                'background': data.get('color', '#d1d5db'),
                'border': '#ffffff' if node_type != 'question' else data.get('color', '#d1d5db'),
            },
            'font': {'color': '#f8fafc' if node_type != 'question' else '#e5e7eb', 'size': font_size, 'face': 'Inter'},
        })
    
    for source, target, data in G.edges(data=True):
        edges_payload.append({
            'from': source,
            'to': target,
            'color': data.get('color', 'rgba(148,163,184,0.18)'),
            'width': data.get('width', 1),
        })
    
    container_id = f"student-graph-{uuid.uuid4().hex}"
    html = f"""
<link rel='stylesheet' href='https://cdnjs.cloudflare.com/ajax/libs/vis-network/9.1.2/dist/dist/vis-network.min.css' />
<script src='https://cdnjs.cloudflare.com/ajax/libs/vis-network/9.1.2/dist/vis-network.min.js'></script>
<div style='background:#050505;border:1px solid #1f2937;border-radius:24px;padding:16px;'>
  <div style='display:flex;justify-content:space-between;align-items:center;gap:16px;margin-bottom:12px;'>
    <div>
      <h3 style='margin:0;color:#f8fafc;font-size:20px;font-weight:700;'>{title}</h3>
      <p style='margin:6px 0 0 0;color:#94a3b8;font-size:13px;'>Yellow hubs are subjects, gray hubs are topics, green question nodes are correct, and red question nodes are wrong.</p>
    </div>
  </div>
  <div style='display:flex;gap:14px;flex-wrap:wrap;margin-bottom:12px;font-size:12px;color:#cbd5e1;'>
    <span><b style='color:#22c55e;'>●</b> Correct</span>
    <span><b style='color:#ef4444;'>●</b> Wrong</span>
    <span><b style='color:#d1d5db;'>●</b> Topic hub</span>
    <span><b style='color:#facc15;'>●</b> Subject hub</span>
  </div>
  <div id='{container_id}' style='height:820px;border-radius:18px;background:#020617;'></div>
</div>
<script>
(() => {{
  const container = document.getElementById('{container_id}');
  if (!container) return;
  const nodes = new vis.DataSet({json.dumps(nodes_payload)});
  const edges = new vis.DataSet({json.dumps(edges_payload)});
  const options = {{
    interaction: {{ hover: true, tooltipDelay: 120, navigationButtons: true, keyboard: true }},
    nodes: {{ borderWidth: 1.2, shadow: {{ enabled: true, color: 'rgba(0,0,0,0.35)', size: 10, x: 0, y: 4 }} }},
    edges: {{ smooth: {{ enabled: true, type: 'dynamic' }} }},
    physics: {{ barnesHut: {{ gravitationalConstant: -9800, springLength: 120, springConstant: 0.028, damping: 0.18, avoidOverlap: 0.45 }}, stabilization: {{ iterations: 420, fit: true }} }}
  }};
  const network = new vis.Network(container, {{ nodes, edges }}, options);
  network.once('stabilized', () => network.fit({{ animation: {{ duration: 700, easingFunction: 'easeInOutQuad' }} }}));
}})();
</script>
"""
    
    Path(output_path).write_text(html, encoding='utf-8')
    display(HTML(html))
    print(f"Saved interactive graph to {output_path}")
    return output_path

def create_simulation_engine(question_perf_df: pd.DataFrame, test_history_df: pd.DataFrame):
    """Create a simulation engine to replay tests"""
    
    print("\n" + "="*60)
    print("🎬 TEST SIMULATION ENGINE")
    print("="*60 + "\n")
    
    if test_history_df.empty:
        print("⚠️ No test data available for simulation")
        return
    
    # Get the latest test
    latest_test = test_history_df.iloc[0]
    latest_test_id = latest_test['id']
    
    # Get questions from this test
    test_questions = question_perf_df[question_perf_df['test_id'] == latest_test_id].sort_values('completed_at')
    
    if test_questions.empty:
        print(f"⚠️ No question data for test {latest_test_id}")
        return
    
    print(f"📝 Test ID: {latest_test_id}")
    print(f"📚 Test Type: {latest_test['test_type']}")
    print(f"📊 Score: {latest_test['score']}/{latest_test['max_score']}")
    print(f"🎯 Questions in test: {len(test_questions)}\n")
    
    # Simulate test progression
    print("Simulating test progression (questions visited in order):\n")
    
    visited_questions = []
    for idx, (_, row) in enumerate(test_questions.iterrows(), 1):
        question_id = row['question_id']
        is_correct = row['correct']
        status = "✅" if is_correct else "❌"
        
        visited_questions.append(question_id)
        
        print(f"  {idx:2}. {question_id:20} {status} {'Correct' if is_correct else 'Wrong'}")
        
        if idx % 10 == 0:
            print()
    
    print(f"\n✅ Simulation complete. {len(visited_questions)} questions visited.")
    print(f"📈 Correct answers: {test_questions['correct'].sum()}/{len(test_questions)}")
    
    return visited_questions

# Create visualizations
if not question_performance.empty:
    # Create graph
    G = create_question_graph(question_performance)
    student_graph = create_student_progress_network(question_performance)
    print(f"\nGraph created with {len(student_graph.nodes())} nodes and {len(student_graph.edges())} edges")
    
    # Visualize
    render_interactive_student_graph(student_graph, "Student Progress Graph (Green=Correct, Red=Wrong)")
    
    # Run simulation
    if not test_history.empty:
        simulation_result = create_simulation_engine(question_performance, test_history)


## Test History and Review System

Browse complete test history, review answers for each test, identify improvement trends, and access detailed performance analytics.


In [86]:
def compare_tests(test_history_df: pd.DataFrame):
    """Compare performance across tests"""
    
    if test_history_df.empty:
        return
    
    test_numbers = list(range(1, len(test_history_df) + 1))
    
    fig = go.Figure()
    
    # Score comparison
    fig.add_trace(go.Scatter(
        x=test_numbers,
        y=test_history_df['score'].tolist(),
        mode='lines+markers',
        name='Score',
        marker=dict(size=10),
        line=dict(width=2, color='#3b82f6')
    ))
    
    # Add accuracy line
    test_history_df['accuracy'] = (test_history_df['correct_answers'] / test_history_df['total_questions'] * 100)
    fig.add_trace(go.Scatter(
        x=test_numbers,
        y=test_history_df['accuracy'].tolist(),
        mode='lines+markers',
        name='Accuracy %',
        marker=dict(size=8),
        line=dict(width=2, color='#10b981', dash='dash'),
        yaxis='y2'
    ))
    
    fig.update_layout(
        title="Test Performance Comparison",
        xaxis_title="Test Number",
        yaxis_title="Score",
        yaxis2=dict(title="Accuracy %", overlaying='y', side='right'),
        hovermode='x unified',
        height=500
    )
    
    fig.show()